# Kappa Foundation | Setup and Data
**BFIN WS2026 — Group Assignment | Group 22**

Decision date: **31 December 2022**. Initial funds: **USD 12 million**. Obligations: **USD 2.4 million (2024)** and **USD 3.6 million (2027)**. The goal is to preserve the December 2022 purchasing power of USD 8 million in 2030.

In [46]:
# 1. Imports, paths and case configuration
import hashlib
import importlib.metadata
import io
import json
import platform
import re
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Notebook is saved at the project root; do not require a fixed folder name.
PROJECT_ROOT = Path.cwd().resolve()
PROJECT_NAME = PROJECT_ROOT.name
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
DOCS_DIR = PROJECT_ROOT / "docs"
for directory in (RAW_DIR, PROCESSED_DIR, TABLES_DIR, DOCS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INITIAL_WEALTH = 12_000_000
DECISION_DATE = pd.Timestamp("2022-12-31")
TERMINAL_DATE = pd.Timestamp("2030-12-31")
PAYMENT_SCHEDULE = {2024: 2_400_000, 2027: 3_600_000}
REAL_TERMINAL_GOAL = 8_000_000
ESTIMATION_START, ESTIMATION_END = "2015-01-01", "2022-12-31"
EVALUATION_START, EVALUATION_END = "2023-01-01", "2025-12-31"
DATA_FREQUENCY, BASE_CURRENCY = "monthly", "USD"

DATA_MODE = "download"  # "offline" or "download"
assert DATA_MODE in {"offline", "download"}
print(f"Project: {PROJECT_ROOT} | Mode: {DATA_MODE}")

Project: E:\University\Studying materials\Year 3\BFIN\Assignment\Kappa_Foundation_1 | Mode: download


In [47]:
# 2. Record the actual working environment and package versions
packages = ["numpy", "pandas", "requests", "yfinance", "scipy", "statsmodels", "matplotlib", "seaborn", "openpyxl", "pyarrow"]
versions = {}
for name in packages:
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        # Optional libraries may not be installed in a data-only environment.
        if name in {"numpy", "pandas", "requests", "yfinance"}:
            raise

environment_info = {
    "project": PROJECT_NAME,
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": versions,
}
(DOCS_DIR / "environment_info.json").write_text(json.dumps(environment_info, indent=2), encoding="utf-8")
(PROJECT_ROOT / "requirements.txt").write_text(
    "".join(f"{name}=={version}\n" for name, version in sorted(versions.items())), encoding="utf-8"
)
print("Environment record and requirements.txt saved.")

Environment record and requirements.txt saved.


## Data sources and assumptions
- Yahoo Finance (`yfinance`) provides historical daily **Adjusting Close** prices for ten US common stocks and two equity ETFs that our team have chosen. The last adjusted close of each month is used to compute simple monthly returns based on this formula: `P_t / P_(t-1) - 1`.
- Kenneth French Data Library supplies monthly Fama–French three-factor data (`Mkt-RF`, `SMB`, `HML`, `RF`). We converted its published percentage units to decimals.

Estimation: **Jan 2015–Dec 2022 (96 months)**. Separate evaluation: **Jan 2023–Dec 2025 (36 months)**.

In [48]:

# Configure data retrieval and raw data storage.


# Define the candidate universe of securities for the Kappa Foundation case study.
SELECTED_STOCKS = ["PG", "PEP", "JNJ", "MRK", "MSFT", "AAPL", "JPM", "CVX", "DE", "NEE"]
SELECTED_ETFS = ["SPY", "IWM"]
ALL_TICKERS = SELECTED_STOCKS + SELECTED_ETFS

DOWNLOAD_START = "2014-12-01"
DOWNLOAD_END_EXCLUSIVE = "2026-01-01"

# Define raw data directories.
YAHOO_RAW_DIR = RAW_DIR / "yahoo_finance"
FRENCH_RAW_DIR = RAW_DIR / "fama_french"

YAHOO_RAW_DIR.mkdir(parents=True, exist_ok=True)
FRENCH_RAW_DIR.mkdir(parents=True, exist_ok=True)



SOURCE_MANIFEST_FILE = DATA_DIR / "source_manifest.json"

def file_sha256(path):
    """Calculate the SHA-256 checksum of a saved file."""
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

# Display the configuration.
print("Data retrieval configuration initialized.")
print(f"Project: {PROJECT_ROOT.name}")
print(f"Data mode: {DATA_MODE}")
print(f"Download start: {DOWNLOAD_START}")
print(f"Download end (exclusive): {DOWNLOAD_END_EXCLUSIVE}")
print(f"Number of securities: {len(ALL_TICKERS)}")
print(f"Tickers: {', '.join(ALL_TICKERS)}")


Data retrieval configuration initialized.
Project: Kappa_Foundation_1
Data mode: download
Download start: 2014-12-01
Download end (exclusive): 2026-01-01
Number of securities: 12
Tickers: PG, PEP, JNJ, MRK, MSFT, AAPL, JPM, CVX, DE, NEE, SPY, IWM


In [49]:
# 4. Download fresh raw prices or read packaged CSV files

def retrieve_yahoo_data(tickers, mode="offline"):
    raw_prices = {}
    failures = {}

    for ticker in tickers:
        file_path = YAHOO_RAW_DIR / f"{ticker}_daily.csv"

        try:
            if mode == "download":
                data = yf.download(
                    tickers=ticker,
                    start=DOWNLOAD_START,
                    end=DOWNLOAD_END_EXCLUSIVE,
                    interval="1d",
                    auto_adjust=False,
                    actions=False,
                    progress=False,
                    threads=False,
                    multi_level_index=False
                )

                if data.empty:
                    raise ValueError("Yahoo Finance returned an empty dataset.")

                if "Adj Close" not in data.columns:
                    raise ValueError("Adjusted Close column is missing.")

                data.to_csv(file_path, index_label="Date")

                time.sleep(0.5)

            elif mode == "offline":
                if not file_path.is_file():
                    raise FileNotFoundError(
                        f"Missing raw file: {file_path.name}"
                    )

                data = pd.read_csv(
                    file_path,
                    index_col="Date",
                    parse_dates=True
                )

            else:
                raise ValueError("Mode must be 'download' or 'offline'.")

            data.index = pd.to_datetime(data.index)
            data = data.sort_index()

            if not data.index.is_unique:
                raise ValueError("Duplicate trading dates detected.")

            raw_prices[ticker] = data

            print(
                f"{ticker}: {len(data)} daily observations loaded."
            )

        except Exception as error:
            failures[ticker] = str(error)
            print(f"{ticker}: FAILED — {error}")

    if failures:
        raise RuntimeError(
            "Data retrieval is incomplete. Resolve failed tickers "
            f"before continuing: {failures}"
        )

    return raw_prices


yahoo_raw = retrieve_yahoo_data(
    ALL_TICKERS,
    mode=DATA_MODE
)

print("\nYahoo Finance retrieval completed successfully.")


PG: 2788 daily observations loaded.
PEP: 2788 daily observations loaded.
JNJ: 2788 daily observations loaded.
MRK: 2788 daily observations loaded.
MSFT: 2788 daily observations loaded.
AAPL: 2788 daily observations loaded.
JPM: 2788 daily observations loaded.
CVX: 2788 daily observations loaded.
DE: 2788 daily observations loaded.
NEE: 2788 daily observations loaded.
SPY: 2788 daily observations loaded.
IWM: 2788 daily observations loaded.

Yahoo Finance retrieval completed successfully.


In [50]:
# 5. Build monthly adjusted prices and simple returns

# Construct monthly adjusted prices from saved daily observations.

monthly_price_series = {}

for ticker, data in yahoo_raw.items():

    adjusted_close = pd.to_numeric(
        data["Adj Close"],
        errors="coerce"
    )

    adjusted_close.index = (
        adjusted_close.index.tz_localize(None)
        if adjusted_close.index.tz is not None
        else adjusted_close.index
    )

    adjusted_close = adjusted_close.dropna()

    monthly_prices = adjusted_close.groupby(
        adjusted_close.index.to_period("M")
    ).last()

    monthly_price_series[ticker] = monthly_prices

monthly_prices = pd.DataFrame(monthly_price_series)

monthly_returns = monthly_prices.pct_change(fill_method=None)

monthly_returns = monthly_returns.loc[
    "2015-01":"2025-12"
]

# Split the data before optimization or estimation to avoid look-ahead bias.
estimation_returns = monthly_returns.loc[
    "2015-01":"2022-12"
].copy()

evaluation_returns = monthly_returns.loc[
    "2023-01":"2025-12"
].copy()

print("Monthly adjusted-price returns constructed.")
print(f"Estimation months: {len(estimation_returns)}")
print(f"Evaluation months: {len(evaluation_returns)}")

display(estimation_returns.head())


Monthly adjusted-price returns constructed.
Estimation months: 96
Evaluation months: 36


,PG,PEP,JNJ,MRK,MSFT,AAPL,JPM,CVX,DE,NEE,SPY,IWM
Date,,,,,,,,,,,,
2015-01,-0.068070,-0.008249,-0.042364,0.061454,-0.130248,0.061424,-0.125442,-0.086023,-0.037075,0.027755,-0.029629,-0.032770
2015-02,0.009966,0.055449,0.030833,-0.028865,0.093121,0.100777,0.126885,0.050698,0.063505,-0.045986,0.056204,0.059464
2015-03,-0.037472,-0.027472,-0.018632,-0.010162,-0.072748,-0.031372,-0.011423,-0.015935,-0.025475,0.005703,-0.015705,0.017700
2015-04,-0.021851,-0.005229,-0.013917,0.036187,0.196262,0.005787,0.051180,0.057916,0.032273,-0.029985,0.009834,-0.025649
2015-05,-0.014086,0.013772,0.016837,0.022330,-0.030334,0.045339,0.039836,-0.063332,0.034910,0.021758,0.012856,0.022363


In [51]:
# 6. Validate and export monthly equity returns
# Verify dates, common observations, and missing returns.

EXPECTED_ESTIMATION_MONTHS = pd.period_range(
    "2015-01", "2022-12", freq="M"
)

EXPECTED_EVALUATION_MONTHS = pd.period_range(
    "2023-01", "2025-12", freq="M"
)

estimation_returns = estimation_returns.reindex(
    EXPECTED_ESTIMATION_MONTHS
)

evaluation_returns = evaluation_returns.reindex(
    EXPECTED_EVALUATION_MONTHS
)

quality_report = pd.DataFrame({
    "Estimation Observations": estimation_returns.count(),
    "Estimation Missing": estimation_returns.isna().sum(),
    "Evaluation Observations": evaluation_returns.count(),
    "Evaluation Missing": evaluation_returns.isna().sum()
})

display(quality_report)

if estimation_returns.isna().any().any():
    raise ValueError(
        "Missing estimation returns detected. "
        "Investigate the raw files before optimization."
    )

if evaluation_returns.isna().any().any():
    raise ValueError(
        "Missing evaluation returns detected. "
        "Investigate the raw files before backtesting."
    )

assert len(estimation_returns) >= 60
assert len(estimation_returns) == 96
assert len(evaluation_returns) == 36

assert (estimation_returns.index <= pd.Period("2022-12")).all()
assert (evaluation_returns.index >= pd.Period("2023-01")).all()

# Save standardized monthly datasets.
monthly_returns.to_csv(
    PROCESSED_DIR / "monthly_equity_returns_full.csv",
    index_label="Month"
)

estimation_returns.to_csv(
    PROCESSED_DIR / "equity_returns_estimation.csv",
    index_label="Month"
)

evaluation_returns.to_csv(
    PROCESSED_DIR / "equity_returns_evaluation.csv",
    index_label="Month"
)

quality_report.to_csv(
    TABLES_DIR / "data_quality_report.csv"
)

print("Return validation passed.")
print("Processed equity datasets saved successfully.")


,Estimation Observations,Estimation Missing,Evaluation Observations,Evaluation Missing
PG,96,0,36,0
PEP,96,0,36,0
JNJ,96,0,36,0
MRK,96,0,36,0
MSFT,96,0,36,0
AAPL,96,0,36,0
JPM,96,0,36,0
CVX,96,0,36,0
DE,96,0,36,0
NEE,96,0,36,0


Return validation passed.
Processed equity datasets saved successfully.


In [52]:
# 7. Retrieve / read raw Kenneth French ZIP and parse monthly factors


FACTOR_URL = (
    "https://mba.tuck.dartmouth.edu/pages/"
    "faculty/ken.french/ftp/"
    "F-F_Research_Data_Factors_CSV.zip"
)

FACTOR_ZIP_PATH = (
    FRENCH_RAW_DIR / "F-F_Research_Data_Factors_CSV.zip"
)

if DATA_MODE == "download":

    response = requests.get(
        FACTOR_URL,
        timeout=60
    )
    response.raise_for_status()

    # Save the original downloaded ZIP without modification.
    FACTOR_ZIP_PATH.write_bytes(response.content)

    print("Fama-French factor archive downloaded.")

elif DATA_MODE == "offline":

    if not FACTOR_ZIP_PATH.exists():
        raise FileNotFoundError(
            "Saved Fama-French ZIP archive was not found."
        )

    print("Using the saved Fama-French factor archive.")

else:
    raise ValueError("Invalid DATA_MODE.")

# Extract the monthly section from the original archive.
with zipfile.ZipFile(FACTOR_ZIP_PATH, "r") as archive:

    csv_files = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    if len(csv_files) != 1:
        raise ValueError(
            f"Expected one factor CSV, found {csv_files}"
        )

    csv_text = archive.read(csv_files[0]).decode(
        "utf-8-sig",
        errors="replace"
    )

factor_rows = []

for line in csv_text.splitlines():

    fields = [field.strip() for field in line.split(",")]

    # Monthly rows start with YYYYMM.
    if not re.fullmatch(r"\d{6}", fields[0]):
        continue

    if len(fields) < 5:
        continue

    try:
        values = [float(value) for value in fields[1:5]]
    except ValueError:
        continue

    factor_rows.append([fields[0], *values])

factors = pd.DataFrame(
    factor_rows,
    columns=["Month", "Mkt-RF", "SMB", "HML", "RF"]
)

if factors.empty:
    raise ValueError("No monthly factor observations parsed.")

if factors["Month"].duplicated().any():
    raise ValueError("Duplicate factor months detected.")

factors["Month"] = pd.PeriodIndex(
    factors["Month"],
    freq="M"
)

factors = factors.set_index("Month").sort_index()

# French monthly factor returns are published in percent.
# Convert all columns to decimal units.
factors = factors / 100.0

factors = factors.loc["2015-01":"2025-12"]

print("Fama-French factor data parsed.")
display(factors.head())


Fama-French factor archive downloaded.
Fama-French factor data parsed.


,Mkt-RF,SMB,HML,RF
Month,,,,
2015-01,-0.031000,-0.006100,-0.034800,0.000000
2015-02,0.061300,0.006400,-0.017500,0.000000
2015-03,-0.011100,0.030600,-0.003800,0.000000
2015-04,0.005900,-0.029800,0.018100,0.000000
2015-05,0.013700,0.009600,-0.011000,0.000000


In [53]:
# 8. Align factors and equity returns, and export model inputs
# Align factor observations with the monthly equity dataset.

required_factor_columns = ["Mkt-RF", "SMB", "HML", "RF"]

factor_estimation = factors.reindex(
    EXPECTED_ESTIMATION_MONTHS
)[required_factor_columns].copy()

factor_evaluation = factors.reindex(
    EXPECTED_EVALUATION_MONTHS
)[required_factor_columns].copy()

if factor_estimation.isna().any().any():
    raise ValueError("Missing estimation factor observations.")

if factor_evaluation.isna().any().any():
    raise ValueError("Missing evaluation factor observations.")

assert factor_estimation.index.equals(
    estimation_returns.index
)

assert factor_evaluation.index.equals(
    evaluation_returns.index
)

# Save factor data independently of equity returns.
factors.to_csv(
    PROCESSED_DIR / "ff3_factors_full.csv",
    index_label="Month"
)

factor_estimation.to_csv(
    PROCESSED_DIR / "ff3_factors_estimation.csv",
    index_label="Month"
)

factor_evaluation.to_csv(
    PROCESSED_DIR / "ff3_factors_evaluation.csv",
    index_label="Month"
)

combined_estimation = estimation_returns.join(
    factor_estimation,
    how="inner",
    validate="one_to_one"
)

combined_evaluation = evaluation_returns.join(
    factor_evaluation,
    how="inner",
    validate="one_to_one"
)

combined_estimation.to_csv(
    PROCESSED_DIR / "combined_estimation_data.csv",
    index_label="Month"
)

combined_evaluation.to_csv(
    PROCESSED_DIR / "combined_evaluation_data.csv",
    index_label="Month"
)

print("Equity and factor data aligned.")
print(f"Estimation observations: {len(combined_estimation)}")
print(f"Evaluation observations: {len(combined_evaluation)}")

display(combined_estimation.head())


Equity and factor data aligned.
Estimation observations: 96
Evaluation observations: 36


,PG,PEP,JNJ,MRK,MSFT,AAPL,JPM,CVX,DE,NEE,SPY,IWM,Mkt-RF,SMB,HML,RF
2015-01,-0.068070,-0.008249,-0.042364,0.061454,-0.130248,0.061424,-0.125442,-0.086023,-0.037075,0.027755,-0.029629,-0.032770,-0.031000,-0.006100,-0.034800,0.000000
2015-02,0.009966,0.055449,0.030833,-0.028865,0.093121,0.100777,0.126885,0.050698,0.063505,-0.045986,0.056204,0.059464,0.061300,0.006400,-0.017500,0.000000
2015-03,-0.037472,-0.027472,-0.018632,-0.010162,-0.072748,-0.031372,-0.011423,-0.015935,-0.025475,0.005703,-0.015705,0.017700,-0.011100,0.030600,-0.003800,0.000000
2015-04,-0.021851,-0.005229,-0.013917,0.036187,0.196262,0.005787,0.051180,0.057916,0.032273,-0.029985,0.009834,-0.025649,0.005900,-0.029800,0.018100,0.000000
2015-05,-0.014086,0.013772,0.016837,0.022330,-0.030334,0.045339,0.039836,-0.063332,0.034910,0.021758,0.012856,0.022363,0.013700,0.009600,-0.011000,0.000000


In [54]:
# 9. Preserve raw-data provenance and verify final integrity

# Preserve download provenance independently of processed data.

if DATA_MODE == "download":
    retrieval_time = datetime.now(timezone.utc).isoformat()

    source_manifest = {
        "project": PROJECT_NAME,
        "original_decision_date": "2022-12-31",
        "retrieval_timestamp_utc": retrieval_time,
        "sources": {
            "yahoo_finance": {
                "provider": "Yahoo Finance",
                "library": "yfinance",
                "library_version": yf.__version__,
                "tickers": ALL_TICKERS,
                "requested_start": DOWNLOAD_START,
                "requested_end_exclusive": DOWNLOAD_END_EXCLUSIVE,
                "interval": "1d",
                "auto_adjust": False,
                "actions": False,
                "price_field": "Adj Close",
                "files": {
                    ticker: {
                        "path": str(
                            (YAHOO_RAW_DIR / f"{ticker}_daily.csv")
                            .relative_to(PROJECT_ROOT)
                        ),
                        "sha256": file_sha256(
                            YAHOO_RAW_DIR / f"{ticker}_daily.csv"
                        )
                    }
                    for ticker in ALL_TICKERS
                }
            },
            "fama_french": {
                "provider": "Kenneth R. French Data Library",
                "dataset": "F-F Research Data Factors",
                "url": FACTOR_URL,
                "file": str(
                    FACTOR_ZIP_PATH.relative_to(PROJECT_ROOT)
                ),
                "sha256": file_sha256(FACTOR_ZIP_PATH),
                "original_units": "percent",
                "processed_units": "decimal"
            }
        },
        "processing": {
            "return_method": "simple monthly return",
            "equity_price_field": "Adj Close",
            "monthly_price_rule": "last available trading day",
            "missing_data_policy": "no imputation",
            "estimation_period": [
                ESTIMATION_START, ESTIMATION_END
            ],
            "evaluation_period": [
                EVALUATION_START, EVALUATION_END
            ]
        }
    }

    with SOURCE_MANIFEST_FILE.open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(source_manifest, file, indent=4)

else:
    if not SOURCE_MANIFEST_FILE.is_file():
        raise FileNotFoundError(
            "Source manifest is missing in offline mode."
        )

    with SOURCE_MANIFEST_FILE.open(
        "r", encoding="utf-8"
    ) as file:
        source_manifest = json.load(file)

    # Verify that offline raw files match the recorded downloads.
    for ticker, record in source_manifest["sources"][
        "yahoo_finance"
    ]["files"].items():

        path = PROJECT_ROOT / record["path"]

        if file_sha256(path) != record["sha256"]:
            raise ValueError(
                f"Raw file checksum mismatch: {ticker}"
            )

    french_record = source_manifest["sources"]["fama_french"]
    french_path = PROJECT_ROOT / french_record["file"]

    if file_sha256(french_path) != french_record["sha256"]:
        raise ValueError("Factor archive checksum mismatch.")

# Final essential integrity checks; avoid duplicate verbose validation tables.
assert len(yahoo_raw) == len(ALL_TICKERS)
assert len(estimation_returns) == 96 and len(evaluation_returns) == 36
assert not combined_estimation.isna().any().any()
assert not combined_evaluation.isna().any().any()
assert combined_estimation.index.equals(estimation_returns.index)
assert combined_evaluation.index.equals(evaluation_returns.index)
assert estimation_returns.index.max() == pd.Period("2022-12")
print("Data validated. Source manifest:", SOURCE_MANIFEST_FILE.relative_to(PROJECT_ROOT))


Data validated. Source manifest: data\source_manifest.json
